In [ ]:

import os, math, random, urllib.request
from pathlib import Path
import numpy as np, cv2, torch, torch.nn as nn
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

# ---------- SETTINGS ----------
DATA_DIR      = "archive/asl_alphabet_train/asl_alphabet_train"
TEST_DIR      = "archive/asl_alphabet_test/asl_alphabet_test"
TASK_PATH     = "hand_landmarker.task"
OUT           = "sign_landmark.pt"
CACHE         = "landmarks_cache.npz"
MAX_PER_CLASS = 800      
EPOCHS        = 60
BATCH         = 128
LR            = 3e-3
VAL_SPLIT     = 0.15




def make_landmarker(conf=0.3):
    opts = vision.HandLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=TASK_PATH),
        running_mode=vision.RunningMode.IMAGE, num_hands=1,
        min_hand_detection_confidence=conf,
        min_hand_presence_confidence=conf,
        min_tracking_confidence=conf)
    return vision.HandLandmarker.create_from_options(opts)

def normalize(pts):
    """(21,3) -> wrist at origin, scaled so the farthest point has length 1"""
    pts = pts - pts[0]
    return (pts / (np.linalg.norm(pts, axis=1).max() + 1e-6)).astype(np.float32)

def extract(landmarker, path):
    img = cv2.imread(str(path))
    if img is None:
        return None
    h, w = img.shape[:2]
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    if not res.hand_landmarks:
        return None
    pts = np.array([[p.x * w, p.y * h, p.z * w] for p in res.hand_landmarks[0]], dtype=np.float32)
    return normalize(pts)


classes = sorted(d.name for d in Path(DATA_DIR).iterdir()
                 if d.is_dir() and d.name.lower() != "nothing")   # "nothing" = no hand detected at runtime
if os.path.exists(CACHE):
    z = np.load(CACHE, allow_pickle=True)
    if list(z["classes"]) == classes and int(z["maxn"]) == (MAX_PER_CLASS or -1):
        X, Y = z["X"], z["Y"]
        print("Loaded cached landmarks:", X.shape)
    else:
        os.remove(CACHE)
if not os.path.exists(CACHE):
    lm = make_landmarker()
    X, Y = [], []
    random.seed(0)
    for ci, c in enumerate(classes):
        files = [f for f in (Path(DATA_DIR) / c).iterdir()
                 if f.suffix.lower() in (".jpg", ".jpeg", ".png")]
        random.shuffle(files)
        if MAX_PER_CLASS:
            files = files[:MAX_PER_CLASS]
        ok = 0
        for f in files:
            p = extract(lm, f)
            if p is not None:
                X.append(p); Y.append(ci); ok += 1
        print(f"{c:>8}: hand found in {ok}/{len(files)} images")
    X, Y = np.stack(X), np.array(Y)
    np.savez(CACHE, X=X, Y=Y, classes=np.array(classes), maxn=(MAX_PER_CLASS or -1))
    print("Saved cache:", X.shape)

# ---------- 2) split ----------
X_t, Y_t = torch.tensor(X), torch.tensor(Y)
g = torch.Generator().manual_seed(42)
tr, va = [], []
for c in range(len(classes)):
    ids = (Y_t == c).nonzero().flatten()
    ids = ids[torch.randperm(len(ids), generator=g)]
    n = max(1, int(len(ids) * VAL_SPLIT))
    va += ids[:n].tolist(); tr += ids[n:].tolist()
Xtr, Ytr, Xva, Yva = X_t[tr], Y_t[tr], X_t[va], Y_t[va]

# ---------- 3) model + augmentation ----------
class LandmarkMLP(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(63, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes))
    def forward(self, x):
        return self.net(x)

def augment(x):
    """random rotation, scale, left/right mirror, noise -> works for either hand / any camera"""
    B = x.shape[0]
    a = (torch.rand(B) * 2 - 1) * math.radians(15)
    c, s, zero, one = torch.cos(a), torch.sin(a), torch.zeros(B), torch.ones(B)
    R = torch.stack([torch.stack([c, -s, zero], 1),
                     torch.stack([s,  c, zero], 1),
                     torch.stack([zero, zero, one], 1)], 1)
    x = x @ R.transpose(1, 2)
    flip = (torch.rand(B) < 0.5).float() * -2 + 1
    x = x * torch.stack([flip, one, one], 1).unsqueeze(1)
    x = x * (0.9 + 0.2 * torch.rand(B, 1, 1))
    return x + 0.01 * torch.randn_like(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = LandmarkMLP(len(classes)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
steps = math.ceil(len(Xtr) / BATCH)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=steps)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

# ---------- 4) train ----------
best = 0.0
print(f"\nTraining on {len(Xtr)} samples, validating on {len(Xva)} | device: {device}")
for ep in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(len(Xtr))
    tot = 0.0
    for i in range(0, len(Xtr), BATCH):
        idx = perm[i:i + BATCH]
        if len(idx) < 2:
            continue
        xb, yb = augment(Xtr[idx]).to(device), Ytr[idx].to(device)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward(); opt.step(); sched.step()
        tot += loss.item() * len(idx)
    model.eval()
    with torch.no_grad():
        acc = (model(Xva.to(device)).argmax(1).cpu() == Yva).float().mean().item()
    if ep % 5 == 0 or ep == 1:
        print(f"Epoch {ep:02d}/{EPOCHS}  loss {tot/len(Xtr):.4f}  val_acc {acc:.4f}")
    if acc >= best:
        best = acc
        torch.save({"model": model.state_dict(), "classes": classes}, OUT)
print(f"Done. Best val acc {best:.4f} -> saved to {OUT}")

# ---------- 5) optional test-folder score ----------
def collect_test(root):
    items = []
    if not os.path.isdir(root):
        return items
    for f in sorted(Path(root).rglob("*")):
        if f.suffix.lower() in (".jpg", ".jpeg", ".png"):
            name = f.parent.name if f.parent != Path(root) else f.stem.split("_")[0]
            if name in classes:
                items.append((f, classes.index(name)))
    return items

test_items = collect_test(TEST_DIR)
if test_items:
    model.load_state_dict(torch.load(OUT, map_location=device)["model"]); model.eval()
    lm = make_landmarker(); ok = n = 0
    for f, y in test_items:
        p = extract(lm, f)
        if p is None:
            n += 1; continue          # counts as a miss
        with torch.no_grad():
            ok += int(model(torch.tensor(p).unsqueeze(0).to(device)).argmax(1).item() == y)
        n += 1
    print(f"Test accuracy: {ok/n:.4f} on {n} images (hand not found = wrong)")
else:
    print("No usable test folder, skipped.")

In [ ]:
# : RUN 
import time, math
from collections import deque, Counter
import numpy as np, cv2, torch, torch.nn as nn
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

# ---------- SETTINGS ----------
TASK_PATH = "hand_landmarker.task"
MODEL     = "sign_landmark.pt"
CAMERA    = 0
CONF      = 0.50     # min confidence to accept a prediction
HOLD      = 20       # frames a sign must be held to be typed
SMOOTH    = 10     

class LandmarkMLP(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(63, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes))
    def forward(self, x):
        return self.net(x)

def normalize(pts):
    pts = pts - pts[0]
    return (pts / (np.linalg.norm(pts, axis=1).max() + 1e-6)).astype(np.float32)

CONNECTIONS = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(5,9),(9,10),(10,11),(11,12),
               (9,13),(13,14),(14,15),(15,16),(13,17),(17,18),(18,19),(19,20),(0,17)]

device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(MODEL, map_location=device)
classes = ckpt["classes"]
model = LandmarkMLP(len(classes)).to(device)
model.load_state_dict(ckpt["model"]); model.eval()

landmarker = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=TASK_PATH),
    running_mode=vision.RunningMode.VIDEO, num_hands=1,
    min_hand_detection_confidence=0.5, min_hand_presence_confidence=0.5, min_tracking_confidence=0.5))

cap = cv2.VideoCapture(CAMERA)
if not cap.isOpened():
    raise SystemExit("Cannot open camera. Try CAMERA = 1")

history = deque(maxlen=SMOOTH)
text, cand, hold, armed = "", None, 0, True
t0, last_ts = time.time(), -1

while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame = cv2.flip(frame, 1)                      # mirror view (model is mirror-invariant)
    h, w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    ts = max(int((time.time() - t0) * 1000), last_ts + 1); last_ts = ts
    res = landmarker.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), ts)

    label, conf = "nothing", 0.0
    if res.hand_landmarks:
        hl = res.hand_landmarks[0]
        pts = np.array([[p.x * w, p.y * h, p.z * w] for p in hl], dtype=np.float32)
        with torch.no_grad():
            probs = torch.softmax(model(torch.tensor(normalize(pts)).unsqueeze(0).to(device)), 1)[0]
            c, i = probs.max(0)
            label, conf = classes[i.item()], c.item()
        px = [(int(p.x * w), int(p.y * h)) for p in hl]
        for a, b in CONNECTIONS:
            cv2.line(frame, px[a], px[b], (0, 255, 0), 2)
        for p in px:
            cv2.circle(frame, p, 4, (0, 0, 255), -1)
        cv2.putText(frame, f"{label} ({conf*100:.0f}%)", (px[0][0], max(px[0][1] - 20, 30)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    # smoothing: majority vote
    history.append(label if (res.hand_landmarks and conf >= CONF) else None)
    top = Counter(l for l in history if l).most_common(1)
    stable = top[0][0] if top and top[0][1] >= SMOOTH * 0.7 else None

    # type the letter once held steadily
    if stable is not None and stable == cand:
        hold += 1
    else:
        cand, hold, armed = stable, 0, True
    if hold == HOLD and armed:
        name = cand.lower()
        if name == "space":
            text += " "
        elif name in ("del", "delete"):
            text = text[:-1]
        else:
            text += cand
        armed = False

    cv2.rectangle(frame, (0, h - 60), (w, h), (0, 0, 0), -1)
    cv2.putText(frame, "Text: " + text, (10, h - 20), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    cv2.imshow("Sign Language (MediaPipe)", frame)

    k = cv2.waitKey(1) & 0xFF
    if k == ord("q"):
        break
    elif k == ord("c"):
        text = ""
    elif k == ord("b"):
        text = text[:-1]
    elif k == 32:
        text += " "

cap.release(); cv2.destroyAllWindows(); landmarker.close()